[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/volume2_chapter_01/notebook_1A_drift_detection.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 1A: Drift Detection and Performance Monitoring

**Volume 2, Chapter 1: Clinical Validation and Evidence**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement statistical tests for detecting distribution drift (KS test, PSI, chi-squared)
2. Monitor model performance over time with appropriate metrics
3. Build interactive monitoring dashboards for deployed AI systems
4. Simulate distribution shifts and observe detection method responses
5. Design alerting thresholds appropriate for different clinical contexts

## Clinical Context

Deployed healthcare AI systems face a fundamental challenge: the world changes, but models do not automatically adapt. Patient populations shift, clinical practices evolve, and data collection methods change. Without systematic monitoring, these changes can silently degrade model performance, potentially harming patients before anyone notices.

**Key insight from Chapter 1:** Regulatory clearance establishes that a model worked on validation data at a specific point in time. It does NOT guarantee ongoing performance. Drift detection is your early warning system.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Define consistent color scheme
COLORS = {
    'reference': '#2E86AB',
    'current': '#E94F37',
    'alert': '#F39C12',
    'safe': '#27AE60'
}

print("✓ Libraries imported successfully")

---

## 1. Generate Simulated Deployment Data

We simulate a sepsis prediction model deployed in an ICU setting. The model uses vital signs and lab values to predict sepsis risk. We generate:
- **Reference period**: Data from model validation (stable, well-characterized)
- **Deployment periods**: Data from subsequent months with various drift patterns

In [ ]:
def generate_sepsis_data(n_patients, drift_type='none', drift_magnitude=0.0, seed=None):
    """
    Generate simulated sepsis prediction data with optional drift.
    
    Parameters:
    -----------
    n_patients : int
        Number of patients to generate
    drift_type : str
        Type of drift: 'none', 'covariate', 'concept', 'prevalence', 'sudden'
    drift_magnitude : float
        Magnitude of drift (0.0 to 1.0)
    seed : int, optional
        Random seed for reproducibility
    
    Returns:
    --------
    pd.DataFrame with features and sepsis labels
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Base feature distributions (reference population)
    age = np.random.normal(65, 12, n_patients)
    age = np.clip(age, 18, 95)
    
    heart_rate = np.random.normal(85, 15, n_patients)
    resp_rate = np.random.normal(18, 4, n_patients)
    temperature = np.random.normal(37.0, 0.5, n_patients)
    sbp = np.random.normal(120, 20, n_patients)
    wbc = np.random.normal(8.5, 3.0, n_patients)
    lactate = np.random.exponential(1.2, n_patients)
    
    # Apply drift based on type
    if drift_type == 'covariate':
        # Population shift: older, sicker patients
        age += drift_magnitude * 10
        heart_rate += drift_magnitude * 8
        lactate += drift_magnitude * 0.5
        
    elif drift_type == 'sudden':
        # Sudden shift (e.g., pandemic-like event)
        resp_rate += drift_magnitude * 6
        temperature += drift_magnitude * 0.3
        wbc += drift_magnitude * 4
    
    # Calculate sepsis probability based on features
    # This represents the "true" underlying relationship
    logit = (
        -6.0 +
        0.03 * (age - 65) +
        0.04 * (heart_rate - 85) +
        0.08 * (resp_rate - 18) +
        0.8 * (temperature - 37.0) +
        -0.02 * (sbp - 120) +
        0.15 * (wbc - 8.5) +
        0.5 * lactate
    )
    
    # Apply concept drift: change the relationship
    if drift_type == 'concept':
        # New treatment protocol reduces lactate importance
        logit = logit - drift_magnitude * 0.3 * lactate
        # Temperature becomes more predictive
        logit = logit + drift_magnitude * 0.5 * (temperature - 37.0)
    
    prob_sepsis = 1 / (1 + np.exp(-logit))
    
    # Apply prevalence drift
    if drift_type == 'prevalence':
        # Increase base rate
        prob_sepsis = prob_sepsis + drift_magnitude * 0.1
        prob_sepsis = np.clip(prob_sepsis, 0, 1)
    
    # Generate binary outcomes
    sepsis = (np.random.random(n_patients) < prob_sepsis).astype(int)
    
    return pd.DataFrame({
        'age': age,
        'heart_rate': heart_rate,
        'resp_rate': resp_rate,
        'temperature': temperature,
        'sbp': sbp,
        'wbc': wbc,
        'lactate': lactate,
        'prob_sepsis': prob_sepsis,
        'sepsis': sepsis
    })

# Generate reference data (validation period)
df_reference = generate_sepsis_data(2000, drift_type='none', seed=42)

print(f"Reference dataset: {len(df_reference)} patients")
print(f"Sepsis prevalence: {df_reference['sepsis'].mean():.1%}")
print(f"\nFeature summary:")
df_reference.describe().round(2)

### Train a Simple Prediction Model

We train a logistic regression model on the reference data. This model represents a deployed system that we will monitor for drift.

In [ ]:
# Define feature columns
feature_cols = ['age', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'wbc', 'lactate']

# Prepare training data
X_ref = df_reference[feature_cols]
y_ref = df_reference['sepsis']

# Standardize features
scaler = StandardScaler()
X_ref_scaled = scaler.fit_transform(X_ref)

# Train logistic regression
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_ref_scaled, y_ref)

# Evaluate on reference data
y_pred_prob_ref = model.predict_proba(X_ref_scaled)[:, 1]
auc_ref = roc_auc_score(y_ref, y_pred_prob_ref)

print(f"Model trained on reference data")
print(f"Reference AUROC: {auc_ref:.3f}")
print(f"\nFeature coefficients:")
for feat, coef in zip(feature_cols, model.coef_[0]):
    print(f"  {feat}: {coef:.3f}")

---

## 2. Statistical Tests for Drift Detection

### 2.1 Kolmogorov-Smirnov (KS) Test

The KS test compares two distributions by measuring the maximum difference between their cumulative distribution functions. It is sensitive to any type of distributional difference (location, spread, shape).

In [ ]:
def ks_test_features(df_ref, df_current, feature_cols, alpha=0.05):
    """
    Perform KS test on all features between reference and current data.
    
    Returns DataFrame with KS statistics and drift flags.
    """
    results = []
    
    for col in feature_cols:
        ref_values = df_ref[col].dropna()
        curr_values = df_current[col].dropna()
        
        # Perform KS test
        ks_stat, p_value = stats.ks_2samp(ref_values, curr_values)
        
        results.append({
            'feature': col,
            'ks_statistic': ks_stat,
            'p_value': p_value,
            'drift_detected': p_value < alpha,
            'ref_mean': ref_values.mean(),
            'ref_std': ref_values.std(),
            'curr_mean': curr_values.mean(),
            'curr_std': curr_values.std(),
            'mean_shift': curr_values.mean() - ref_values.mean()
        })
    
    return pd.DataFrame(results)

# Generate current data with covariate drift
df_current_drift = generate_sepsis_data(500, drift_type='covariate', drift_magnitude=0.5, seed=100)

# Perform KS tests
ks_results = ks_test_features(df_reference, df_current_drift, feature_cols)

print("KS Test Results (Reference vs. Drifted Data)")
print("=" * 80)
print(ks_results.to_string(index=False))

In [ ]:
# Visualize KS test for a drifted feature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution comparison
ax = axes[0]
ax.hist(df_reference['age'], bins=30, alpha=0.6, density=True, 
        label='Reference', color=COLORS['reference'])
ax.hist(df_current_drift['age'], bins=30, alpha=0.6, density=True, 
        label='Current (Drifted)', color=COLORS['current'])
ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Age Distribution: Covariate Drift Detected', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

# CDF comparison with KS statistic
ax = axes[1]
ref_sorted = np.sort(df_reference['age'])
curr_sorted = np.sort(df_current_drift['age'])
ref_cdf = np.arange(1, len(ref_sorted) + 1) / len(ref_sorted)
curr_cdf = np.arange(1, len(curr_sorted) + 1) / len(curr_sorted)

ax.plot(ref_sorted, ref_cdf, label='Reference CDF', color=COLORS['reference'], linewidth=2)
ax.plot(curr_sorted, curr_cdf, label='Current CDF', color=COLORS['current'], linewidth=2)
ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Cumulative Probability', fontsize=12)
ax.set_title('Cumulative Distribution Functions', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

# Add KS statistic annotation
ks_stat = ks_results[ks_results['feature'] == 'age']['ks_statistic'].values[0]
ax.annotate(f'KS Statistic = {ks_stat:.3f}', xy=(0.6, 0.2), xycoords='axes fraction',
            fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### 2.2 Population Stability Index (PSI)

PSI is widely used in credit scoring and healthcare to measure population shift. It compares the distribution of a variable across bins between reference and current periods.

**Interpretation:**
- PSI < 0.10: No significant change
- 0.10 ≤ PSI < 0.25: Moderate change, investigate
- PSI ≥ 0.25: Significant change, action required

In [ ]:
def calculate_psi(reference, current, n_bins=10):
    """
    Calculate Population Stability Index.
    
    Parameters:
    -----------
    reference : array-like
        Reference distribution values
    current : array-like
        Current distribution values
    n_bins : int
        Number of bins for discretization
    
    Returns:
    --------
    float : PSI value
    pd.DataFrame : Bin-level details
    """
    # Create bins based on reference distribution
    bins = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
    bins[0] = -np.inf
    bins[-1] = np.inf
    
    # Calculate proportions in each bin
    ref_counts, _ = np.histogram(reference, bins=bins)
    curr_counts, _ = np.histogram(current, bins=bins)
    
    # Convert to proportions (add small constant to avoid division by zero)
    ref_pct = (ref_counts + 0.0001) / len(reference)
    curr_pct = (curr_counts + 0.0001) / len(current)
    
    # Calculate PSI for each bin
    psi_values = (curr_pct - ref_pct) * np.log(curr_pct / ref_pct)
    psi_total = np.sum(psi_values)
    
    # Create detailed breakdown
    details = pd.DataFrame({
        'bin': range(1, n_bins + 1),
        'ref_pct': ref_pct * 100,
        'curr_pct': curr_pct * 100,
        'psi_contribution': psi_values
    })
    
    return psi_total, details

def calculate_psi_all_features(df_ref, df_current, feature_cols):
    """
    Calculate PSI for all features.
    """
    results = []
    
    for col in feature_cols:
        psi, _ = calculate_psi(df_ref[col].values, df_current[col].values)
        
        # Determine severity
        if psi < 0.10:
            status = 'OK'
        elif psi < 0.25:
            status = 'INVESTIGATE'
        else:
            status = 'ACTION REQUIRED'
        
        results.append({
            'feature': col,
            'psi': psi,
            'status': status
        })
    
    return pd.DataFrame(results)

# Calculate PSI for drifted data
psi_results = calculate_psi_all_features(df_reference, df_current_drift, feature_cols)

print("Population Stability Index Results")
print("=" * 60)
print(psi_results.to_string(index=False))
print("\nThresholds: PSI < 0.10 = OK, 0.10-0.25 = Investigate, ≥0.25 = Action Required")

In [ ]:
# Visualize PSI breakdown for age
psi_age, psi_details = calculate_psi(df_reference['age'].values, df_current_drift['age'].values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of bin proportions
ax = axes[0]
x = np.arange(len(psi_details))
width = 0.35
ax.bar(x - width/2, psi_details['ref_pct'], width, label='Reference', color=COLORS['reference'])
ax.bar(x + width/2, psi_details['curr_pct'], width, label='Current', color=COLORS['current'])
ax.set_xlabel('Bin', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Age Distribution by Bin', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.legend(fontsize=11)

# PSI contribution by bin
ax = axes[1]
colors = [COLORS['alert'] if v > 0.02 else COLORS['safe'] for v in psi_details['psi_contribution']]
ax.bar(psi_details['bin'], psi_details['psi_contribution'], color=colors)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Bin', fontsize=12)
ax.set_ylabel('PSI Contribution', fontsize=12)
ax.set_title(f'PSI Contribution by Bin (Total PSI = {psi_age:.3f})', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 2.3 Chi-Squared Test for Categorical Variables

For categorical features, the chi-squared test compares observed frequencies against expected frequencies based on the reference distribution.

In [ ]:
def chi_squared_test_categorical(ref_values, curr_values, alpha=0.05):
    """
    Perform chi-squared test for categorical variable drift.
    """
    # Get unique categories from both
    all_categories = list(set(ref_values) | set(curr_values))
    
    # Count frequencies
    ref_counts = pd.Series(ref_values).value_counts().reindex(all_categories, fill_value=0)
    curr_counts = pd.Series(curr_values).value_counts().reindex(all_categories, fill_value=0)
    
    # Expected counts based on reference proportions
    ref_proportions = ref_counts / ref_counts.sum()
    expected_counts = ref_proportions * curr_counts.sum()
    
    # Chi-squared test
    chi2, p_value = stats.chisquare(curr_counts, f_exp=expected_counts)
    
    return {
        'chi2_statistic': chi2,
        'p_value': p_value,
        'drift_detected': p_value < alpha,
        'ref_proportions': ref_proportions,
        'curr_proportions': curr_counts / curr_counts.sum()
    }

# Create categorical variable: age groups
def create_age_groups(ages):
    return pd.cut(ages, bins=[0, 40, 60, 75, 100], labels=['<40', '40-60', '60-75', '75+'])

ref_age_groups = create_age_groups(df_reference['age'])
curr_age_groups = create_age_groups(df_current_drift['age'])

chi2_result = chi_squared_test_categorical(ref_age_groups, curr_age_groups)

print("Chi-Squared Test for Age Groups")
print("=" * 50)
print(f"Chi-squared statistic: {chi2_result['chi2_statistic']:.2f}")
print(f"P-value: {chi2_result['p_value']:.4f}")
print(f"Drift detected: {chi2_result['drift_detected']}")
print("\nProportions by group:")
comparison = pd.DataFrame({
    'Reference': chi2_result['ref_proportions'] * 100,
    'Current': chi2_result['curr_proportions'] * 100
}).round(1)
print(comparison)

---

## 3. Performance Monitoring Over Time

While statistical tests can detect input drift, the most important measure is whether model performance is changing. This section simulates deployment over multiple time periods with different drift patterns.

In [ ]:
def evaluate_model_performance(model, scaler, df, feature_cols):
    """
    Evaluate model on a dataset and return comprehensive metrics.
    """
    X = df[feature_cols]
    y = df['sepsis']
    
    X_scaled = scaler.transform(X)
    y_pred_prob = model.predict_proba(X_scaled)[:, 1]
    y_pred = (y_pred_prob >= 0.5).astype(int)
    
    return {
        'auroc': roc_auc_score(y, y_pred_prob),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall': recall_score(y, y_pred, zero_division=0),
        'f1': f1_score(y, y_pred, zero_division=0),
        'prevalence': y.mean(),
        'n_patients': len(df)
    }

# Simulate 12 months of deployment with varying drift
deployment_scenarios = [
    {'month': 1, 'drift_type': 'none', 'magnitude': 0.0},
    {'month': 2, 'drift_type': 'none', 'magnitude': 0.0},
    {'month': 3, 'drift_type': 'covariate', 'magnitude': 0.1},
    {'month': 4, 'drift_type': 'covariate', 'magnitude': 0.2},
    {'month': 5, 'drift_type': 'covariate', 'magnitude': 0.3},
    {'month': 6, 'drift_type': 'concept', 'magnitude': 0.2},
    {'month': 7, 'drift_type': 'concept', 'magnitude': 0.4},
    {'month': 8, 'drift_type': 'prevalence', 'magnitude': 0.3},
    {'month': 9, 'drift_type': 'prevalence', 'magnitude': 0.5},
    {'month': 10, 'drift_type': 'sudden', 'magnitude': 0.6},
    {'month': 11, 'drift_type': 'sudden', 'magnitude': 0.4},
    {'month': 12, 'drift_type': 'covariate', 'magnitude': 0.2}
]

monitoring_results = []

for scenario in deployment_scenarios:
    df_month = generate_sepsis_data(
        n_patients=400,
        drift_type=scenario['drift_type'],
        drift_magnitude=scenario['magnitude'],
        seed=scenario['month'] * 100
    )
    
    metrics = evaluate_model_performance(model, scaler, df_month, feature_cols)
    metrics['month'] = scenario['month']
    metrics['drift_type'] = scenario['drift_type']
    metrics['drift_magnitude'] = scenario['magnitude']
    
    # Calculate PSI for this month
    psi_results = calculate_psi_all_features(df_reference, df_month, feature_cols)
    metrics['max_psi'] = psi_results['psi'].max()
    metrics['mean_psi'] = psi_results['psi'].mean()
    
    monitoring_results.append(metrics)

df_monitoring = pd.DataFrame(monitoring_results)
print("Monthly Monitoring Results")
print("=" * 100)
print(df_monitoring.to_string(index=False))

In [ ]:
# Comprehensive monitoring dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# AUROC over time
ax = axes[0, 0]
ax.plot(df_monitoring['month'], df_monitoring['auroc'], 'o-', 
        color=COLORS['reference'], linewidth=2, markersize=8)
ax.axhline(y=auc_ref, color=COLORS['safe'], linestyle='--', 
           label=f'Reference AUROC ({auc_ref:.3f})', linewidth=2)
ax.axhline(y=auc_ref - 0.05, color=COLORS['alert'], linestyle=':', 
           label='Alert Threshold (-0.05)', linewidth=2)
ax.fill_between(df_monitoring['month'], auc_ref - 0.05, 0.5, 
                alpha=0.1, color=COLORS['alert'])
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('AUROC', fontsize=12)
ax.set_title('Model Discrimination Over Time', fontsize=13, fontweight='bold')
ax.legend(loc='lower left', fontsize=10)
ax.set_ylim(0.65, 0.95)
ax.grid(True, alpha=0.3)

# PSI over time
ax = axes[0, 1]
ax.plot(df_monitoring['month'], df_monitoring['max_psi'], 'o-', 
        color=COLORS['current'], linewidth=2, markersize=8, label='Max PSI')
ax.plot(df_monitoring['month'], df_monitoring['mean_psi'], 's--', 
        color=COLORS['reference'], linewidth=2, markersize=6, label='Mean PSI')
ax.axhline(y=0.25, color=COLORS['alert'], linestyle=':', 
           label='Action Threshold (0.25)', linewidth=2)
ax.axhline(y=0.10, color=COLORS['safe'], linestyle=':', 
           label='Investigation Threshold (0.10)', linewidth=2)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('PSI', fontsize=12)
ax.set_title('Population Stability Index Over Time', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

# Precision and Recall
ax = axes[1, 0]
ax.plot(df_monitoring['month'], df_monitoring['precision'], 'o-', 
        color='#8E44AD', linewidth=2, markersize=8, label='Precision')
ax.plot(df_monitoring['month'], df_monitoring['recall'], 's-', 
        color='#16A085', linewidth=2, markersize=8, label='Recall')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision and Recall Over Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Prevalence changes
ax = axes[1, 1]
colors = [COLORS['alert'] if dt != 'none' else COLORS['safe'] 
          for dt in df_monitoring['drift_type']]
ax.bar(df_monitoring['month'], df_monitoring['prevalence'] * 100, color=colors)
ax.axhline(y=df_reference['sepsis'].mean() * 100, color='black', 
           linestyle='--', label='Reference Prevalence', linewidth=2)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Sepsis Prevalence (%)', fontsize=12)
ax.set_title('Outcome Prevalence Over Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add drift type annotations
for i, row in df_monitoring.iterrows():
    if row['drift_type'] != 'none':
        ax.annotate(row['drift_type'][:3], 
                   xy=(row['month'], row['prevalence'] * 100 + 1),
                   ha='center', fontsize=8, rotation=45)

plt.tight_layout()
plt.savefig('monitoring_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Dashboard saved as 'monitoring_dashboard.png'")

---

## 4. Building an Automated Alert System

A production monitoring system needs automated alerts with appropriate thresholds. This section implements a tiered alerting framework.

In [ ]:
class DriftMonitor:
    """
    Comprehensive drift monitoring system with tiered alerts.
    """
    
    def __init__(self, reference_data, feature_cols, model, scaler, reference_auroc):
        self.reference_data = reference_data
        self.feature_cols = feature_cols
        self.model = model
        self.scaler = scaler
        self.reference_auroc = reference_auroc
        
        # Alert thresholds
        self.thresholds = {
            'psi_warning': 0.10,
            'psi_critical': 0.25,
            'auroc_warning': 0.03,  # Relative to reference
            'auroc_critical': 0.07,
            'ks_warning': 0.05,  # p-value threshold
            'prevalence_warning': 0.02,  # Absolute change
            'prevalence_critical': 0.05
        }
        
        self.alert_history = []
    
    def check_drift(self, current_data, period_label='Current'):
        """
        Perform comprehensive drift check and generate alerts.
        """
        alerts = []
        metrics = {}
        
        # 1. Check PSI for each feature
        psi_alerts = []
        for col in self.feature_cols:
            psi, _ = calculate_psi(
                self.reference_data[col].values,
                current_data[col].values
            )
            metrics[f'psi_{col}'] = psi
            
            if psi >= self.thresholds['psi_critical']:
                psi_alerts.append(('CRITICAL', f'{col}: PSI={psi:.3f}'))
            elif psi >= self.thresholds['psi_warning']:
                psi_alerts.append(('WARNING', f'{col}: PSI={psi:.3f}'))
        
        if psi_alerts:
            alerts.extend(psi_alerts)
        
        # 2. Check model performance
        perf = evaluate_model_performance(
            self.model, self.scaler, current_data, self.feature_cols
        )
        metrics['auroc'] = perf['auroc']
        auroc_drop = self.reference_auroc - perf['auroc']
        
        if auroc_drop >= self.thresholds['auroc_critical']:
            alerts.append(('CRITICAL', f"AUROC dropped by {auroc_drop:.3f} (current: {perf['auroc']:.3f})"))
        elif auroc_drop >= self.thresholds['auroc_warning']:
            alerts.append(('WARNING', f"AUROC dropped by {auroc_drop:.3f} (current: {perf['auroc']:.3f})"))
        
        # 3. Check prevalence shift
        ref_prevalence = self.reference_data['sepsis'].mean()
        curr_prevalence = current_data['sepsis'].mean()
        prevalence_change = abs(curr_prevalence - ref_prevalence)
        metrics['prevalence'] = curr_prevalence
        metrics['prevalence_change'] = prevalence_change
        
        if prevalence_change >= self.thresholds['prevalence_critical']:
            alerts.append(('CRITICAL', f"Prevalence shifted: {ref_prevalence:.1%} → {curr_prevalence:.1%}"))
        elif prevalence_change >= self.thresholds['prevalence_warning']:
            alerts.append(('WARNING', f"Prevalence shifted: {ref_prevalence:.1%} → {curr_prevalence:.1%}"))
        
        # 4. Run KS tests
        ks_results = ks_test_features(
            self.reference_data, current_data, self.feature_cols
        )
        n_ks_drift = ks_results['drift_detected'].sum()
        if n_ks_drift >= 3:
            alerts.append(('WARNING', f"KS test detected drift in {n_ks_drift} features"))
        
        # Store results
        result = {
            'period': period_label,
            'alerts': alerts,
            'metrics': metrics,
            'overall_status': self._determine_status(alerts)
        }
        self.alert_history.append(result)
        
        return result
    
    def _determine_status(self, alerts):
        """Determine overall status based on alerts."""
        if any(level == 'CRITICAL' for level, _ in alerts):
            return 'CRITICAL'
        elif any(level == 'WARNING' for level, _ in alerts):
            return 'WARNING'
        else:
            return 'OK'
    
    def print_report(self, result):
        """Print formatted alert report."""
        print("\n" + "=" * 70)
        print(f"DRIFT MONITORING REPORT: {result['period']}")
        print("=" * 70)
        
        status_colors = {'OK': '✅', 'WARNING': '⚠️', 'CRITICAL': '🚨'}
        print(f"\nOverall Status: {status_colors[result['overall_status']]} {result['overall_status']}")
        
        print(f"\nKey Metrics:")
        print(f"  AUROC: {result['metrics']['auroc']:.3f} (Reference: {self.reference_auroc:.3f})")
        print(f"  Prevalence: {result['metrics']['prevalence']:.1%}")
        
        if result['alerts']:
            print(f"\nAlerts ({len(result['alerts'])}):\n")
            for level, message in result['alerts']:
                prefix = '🚨' if level == 'CRITICAL' else '⚠️'
                print(f"  {prefix} [{level}] {message}")
        else:
            print("\n✅ No alerts - all metrics within acceptable ranges")
        
        print("\n" + "=" * 70)

In [ ]:
# Initialize monitor
monitor = DriftMonitor(
    reference_data=df_reference,
    feature_cols=feature_cols,
    model=model,
    scaler=scaler,
    reference_auroc=auc_ref
)

# Test with different scenarios
print("Testing Drift Monitor with Different Scenarios")

# Scenario 1: No drift
df_no_drift = generate_sepsis_data(500, drift_type='none', seed=200)
result1 = monitor.check_drift(df_no_drift, 'Month 1 - Normal Operations')
monitor.print_report(result1)

# Scenario 2: Moderate covariate drift
df_moderate_drift = generate_sepsis_data(500, drift_type='covariate', drift_magnitude=0.4, seed=201)
result2 = monitor.check_drift(df_moderate_drift, 'Month 2 - Population Shift')
monitor.print_report(result2)

# Scenario 3: Severe sudden drift
df_severe_drift = generate_sepsis_data(500, drift_type='sudden', drift_magnitude=0.7, seed=202)
result3 = monitor.check_drift(df_severe_drift, 'Month 3 - Crisis Event')
monitor.print_report(result3)

---

## 5. Simulating Different Drift Patterns

Understanding how different types of drift affect model performance helps you design appropriate monitoring strategies.

In [ ]:
# Compare detection sensitivity across drift types
drift_types = ['none', 'covariate', 'concept', 'prevalence', 'sudden']
magnitudes = [0.0, 0.2, 0.4, 0.6, 0.8]

sensitivity_results = []

for drift_type in drift_types:
    for mag in magnitudes:
        if drift_type == 'none' and mag > 0:
            continue
        if drift_type != 'none' and mag == 0:
            continue
            
        df_test = generate_sepsis_data(500, drift_type=drift_type, drift_magnitude=mag, seed=int(mag*100))
        
        # Calculate metrics
        perf = evaluate_model_performance(model, scaler, df_test, feature_cols)
        psi_results = calculate_psi_all_features(df_reference, df_test, feature_cols)
        ks_results = ks_test_features(df_reference, df_test, feature_cols)
        
        sensitivity_results.append({
            'drift_type': drift_type,
            'magnitude': mag,
            'auroc': perf['auroc'],
            'auroc_drop': auc_ref - perf['auroc'],
            'max_psi': psi_results['psi'].max(),
            'mean_psi': psi_results['psi'].mean(),
            'n_ks_drift': ks_results['drift_detected'].sum()
        })

df_sensitivity = pd.DataFrame(sensitivity_results)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# AUROC drop by drift type and magnitude
ax = axes[0]
for drift_type in ['covariate', 'concept', 'prevalence', 'sudden']:
    subset = df_sensitivity[df_sensitivity['drift_type'] == drift_type]
    ax.plot(subset['magnitude'], subset['auroc_drop'], 'o-', label=drift_type, linewidth=2, markersize=8)
ax.axhline(y=0.05, color='orange', linestyle='--', label='Warning threshold')
ax.axhline(y=0.07, color='red', linestyle='--', label='Critical threshold')
ax.set_xlabel('Drift Magnitude', fontsize=12)
ax.set_ylabel('AUROC Drop', fontsize=12)
ax.set_title('Performance Impact by Drift Type', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Max PSI by drift type
ax = axes[1]
for drift_type in ['covariate', 'concept', 'prevalence', 'sudden']:
    subset = df_sensitivity[df_sensitivity['drift_type'] == drift_type]
    ax.plot(subset['magnitude'], subset['max_psi'], 's-', label=drift_type, linewidth=2, markersize=8)
ax.axhline(y=0.10, color='orange', linestyle='--', label='Warning threshold')
ax.axhline(y=0.25, color='red', linestyle='--', label='Critical threshold')
ax.set_xlabel('Drift Magnitude', fontsize=12)
ax.set_ylabel('Maximum PSI', fontsize=12)
ax.set_title('PSI Detection Sensitivity', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Number of features with KS drift
ax = axes[2]
for drift_type in ['covariate', 'concept', 'prevalence', 'sudden']:
    subset = df_sensitivity[df_sensitivity['drift_type'] == drift_type]
    ax.plot(subset['magnitude'], subset['n_ks_drift'], '^-', label=drift_type, linewidth=2, markersize=8)
ax.set_xlabel('Drift Magnitude', fontsize=12)
ax.set_ylabel('Features with KS Drift', fontsize=12)
ax.set_title('KS Test Detection by Drift Type', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, len(feature_cols) + 0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Key Observations

1. **Covariate drift** is most easily detected by PSI and KS tests (input distribution changes)
2. **Concept drift** causes performance drops but may not trigger input distribution alerts
3. **Prevalence drift** affects calibration more than discrimination (AUROC may remain stable)
4. **Sudden drift** typically triggers all detection methods simultaneously

**Implication:** A robust monitoring system needs multiple complementary detection methods. No single test catches all types of drift.

---

## 6. Designing Context-Appropriate Thresholds

Alert thresholds should be calibrated to the clinical context. A sepsis alert system in a busy ICU cannot tolerate the same false alarm rate as a screening tool in primary care.

In [ ]:
def simulate_alert_burden(monitor, n_periods=50, drift_prob=0.3, seed=42):
    """
    Simulate alert burden under realistic conditions.
    """
    np.random.seed(seed)
    results = []
    
    for period in range(n_periods):
        # Randomly introduce drift
        if np.random.random() < drift_prob:
            drift_type = np.random.choice(['covariate', 'concept', 'prevalence', 'sudden'])
            magnitude = np.random.uniform(0.1, 0.5)
        else:
            drift_type = 'none'
            magnitude = 0.0
        
        df_period = generate_sepsis_data(300, drift_type=drift_type, drift_magnitude=magnitude, seed=period)
        result = monitor.check_drift(df_period, f'Period {period+1}')
        
        results.append({
            'period': period + 1,
            'actual_drift': drift_type != 'none',
            'drift_type': drift_type,
            'magnitude': magnitude,
            'detected_status': result['overall_status'],
            'n_alerts': len(result['alerts']),
            'auroc': result['metrics']['auroc']
        })
    
    return pd.DataFrame(results)

# Reset monitor
monitor = DriftMonitor(
    reference_data=df_reference,
    feature_cols=feature_cols,
    model=model,
    scaler=scaler,
    reference_auroc=auc_ref
)

# Simulate
df_simulation = simulate_alert_burden(monitor, n_periods=100, drift_prob=0.25)

# Calculate performance metrics
true_drift = df_simulation['actual_drift']
detected = df_simulation['detected_status'] != 'OK'

true_positives = (true_drift & detected).sum()
false_positives = (~true_drift & detected).sum()
true_negatives = (~true_drift & ~detected).sum()
false_negatives = (true_drift & ~detected).sum()

sensitivity = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
specificity = true_negatives / (true_negatives + false_positives) if (true_negatives + false_positives) > 0 else 0
ppv = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0

print("Alert System Performance (100 simulated periods)")
print("=" * 50)
print(f"\nActual drift events: {true_drift.sum()}")
print(f"Alerts triggered: {detected.sum()}")
print(f"\nConfusion Matrix:")
print(f"  True Positives: {true_positives}")
print(f"  False Positives: {false_positives}")
print(f"  True Negatives: {true_negatives}")
print(f"  False Negatives: {false_negatives}")
print(f"\nPerformance:")
print(f"  Sensitivity (Drift Detection Rate): {sensitivity:.1%}")
print(f"  Specificity: {specificity:.1%}")
print(f"  Positive Predictive Value: {ppv:.1%}")
print(f"  Alert Rate (all periods): {detected.mean():.1%}")

In [ ]:
# Visualize threshold impact
def evaluate_thresholds(psi_threshold, auroc_threshold):
    """Evaluate detection performance with different thresholds."""
    monitor_test = DriftMonitor(
        reference_data=df_reference,
        feature_cols=feature_cols,
        model=model,
        scaler=scaler,
        reference_auroc=auc_ref
    )
    monitor_test.thresholds['psi_warning'] = psi_threshold
    monitor_test.thresholds['auroc_warning'] = auroc_threshold
    
    df_sim = simulate_alert_burden(monitor_test, n_periods=100, drift_prob=0.25, seed=123)
    
    true_drift = df_sim['actual_drift']
    detected = df_sim['detected_status'] != 'OK'
    
    tp = (true_drift & detected).sum()
    fp = (~true_drift & detected).sum()
    fn = (true_drift & ~detected).sum()
    tn = (~true_drift & ~detected).sum()
    
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return sens, spec, detected.mean()

# Test different threshold combinations
psi_thresholds = [0.05, 0.10, 0.15, 0.20, 0.25]
auroc_thresholds = [0.02, 0.03, 0.05, 0.07, 0.10]

threshold_results = []
for psi_t in psi_thresholds:
    for auroc_t in auroc_thresholds:
        sens, spec, alert_rate = evaluate_thresholds(psi_t, auroc_t)
        threshold_results.append({
            'psi_threshold': psi_t,
            'auroc_threshold': auroc_t,
            'sensitivity': sens,
            'specificity': spec,
            'alert_rate': alert_rate
        })

df_thresholds = pd.DataFrame(threshold_results)

# Create heatmaps
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, metric in enumerate(['sensitivity', 'specificity', 'alert_rate']):
    ax = axes[idx]
    pivot = df_thresholds.pivot(index='psi_threshold', columns='auroc_threshold', values=metric)
    
    cmap = 'RdYlGn' if metric == 'sensitivity' else 'RdYlGn_r' if metric == 'alert_rate' else 'RdYlGn'
    sns.heatmap(pivot, annot=True, fmt='.0%', cmap=cmap, ax=ax, vmin=0, vmax=1)
    ax.set_title(f'{metric.replace("_", " ").title()}', fontsize=13, fontweight='bold')
    ax.set_xlabel('AUROC Threshold', fontsize=11)
    ax.set_ylabel('PSI Threshold', fontsize=11)

plt.tight_layout()
plt.show()

print("\n💡 Key insight: Lower thresholds increase sensitivity but also alert burden.")
print("   Choose thresholds based on your clinical context and resources.")

---

## 7. Summary and Best Practices

### Key Takeaways

1. **Multiple detection methods are essential**: KS tests and PSI detect input drift; performance monitoring detects concept drift; neither alone is sufficient.

2. **Thresholds must be context-appropriate**: A high-stakes ICU system needs sensitive thresholds even at the cost of more alerts. A screening tool may tolerate more missed drift to reduce false alarms.

3. **Tiered alerting reduces noise**: Not every statistical deviation requires immediate action. Use WARNING and CRITICAL tiers appropriately.

4. **Concept drift is hardest to detect**: Changes in the feature-outcome relationship may not trigger input distribution alarms. Regular performance evaluation against ground truth is essential.

5. **Prevalence drift affects calibration more than discrimination**: AUROC may remain stable while predicted probabilities become miscalibrated. Monitor calibration separately.

### Recommended Monitoring Cadence

| Check Type | Frequency | Threshold |
|------------|-----------|----------|
| Input drift (PSI, KS) | Daily/Weekly | PSI > 0.10 warning, > 0.25 critical |
| Performance metrics | Weekly/Monthly | AUROC drop > 0.03 warning, > 0.07 critical |
| Subgroup analysis | Monthly | Any subgroup > 0.05 AUROC drop |
| Calibration | Monthly | Hosmer-Lemeshow p < 0.05 |
| Full audit | Quarterly | Comprehensive review |

### Next Steps

In practice, you would:
1. Integrate this monitoring into your MLOps pipeline
2. Connect alerts to your incident management system
3. Establish clear escalation procedures
4. Document all drift events and responses
5. Use drift patterns to inform retraining decisions

In [ ]:
# Final summary visualization
print("\n" + "="*70)
print("NOTEBOOK COMPLETE: Drift Detection and Performance Monitoring")
print("="*70)
print("""
✅ Implemented KS test for distribution comparison
✅ Implemented PSI for population stability
✅ Built comprehensive monitoring dashboard
✅ Created automated alert system with tiered thresholds
✅ Simulated different drift patterns and evaluated detection sensitivity
✅ Analyzed threshold tradeoffs for different clinical contexts

Remember: Regulatory clearance is necessary but not sufficient.
Continuous monitoring is your safety net for deployed AI systems.
""")

---

## Exercises

1. **Modify thresholds**: Change the alert thresholds in `DriftMonitor` and observe how this affects the detection rate and false alarm rate.

2. **Add calibration monitoring**: Extend the `evaluate_model_performance` function to include Brier score and expected calibration error.

3. **Implement subgroup monitoring**: Add monitoring for performance across age groups, ensuring the model does not degrade for specific populations.

4. **Create a retraining trigger**: Design logic that recommends retraining based on the pattern of alerts over multiple periods.

5. **Real data exercise**: If you have access to clinical data, apply these techniques to monitor a real prediction model over time.

---

*This notebook accompanies Chapter 1 of AI in Healthcare Volume 2. For the full clinical and regulatory context, please refer to the textbook.*